In [1]:
import os
import sys
import torch
from torchvision.transforms import v2
from torch.utils.data import DataLoader, Subset
module_path = os.path.abspath(os.path.join('..', 'src'))
if module_path not in sys.path:
    sys.path.append(module_path)
from dataloader import Dataset

transforms = v2.Compose([
    v2.Resize(256),
    v2.RandomResizedCrop(224),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Original Dataset
dataset = Dataset (
    patches_path="../data/CNRPark-EXT/PATCHES",
    labels_file="../data/CNRPark-EXT/LABELS/train.txt",
    transforms=transforms
)

# Creating new subset with 100 imagens (for testing loop implementation)
indices_teste = list(range(100))
mini_dataset = Subset(dataset, indices_teste)

# Batch size of 32 images
batch_size = 32

train_dataloader = DataLoader(
    mini_dataset, 
    batch_size=32, 
    shuffle=True
)

print(f"Tamanho do dataset original: {len(dataset)}")
print(f"Tamanho do mini-dataset para teste: {len(mini_dataset)}")

Tamanho do dataset original: 94493
Tamanho do mini-dataset para teste: 100


In [2]:
import torch.nn as nn

class mAlex(nn.Module):
    def __init__(self):
        super().__init__()

        # First Layer: Conv + ReLU + LRN + MaxPool
        self.layer1 = nn.Sequential (
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=11, stride=4),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5,alpha=0.0001,beta=0.75,k=1.0),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        # Second Layer: Conv + ReLU + LRN + MaxPool
        self.layer2 = nn.Sequential (
            nn.Conv2d(in_channels=16, out_channels=20, kernel_size=5, stride=1),
            nn.ReLU(inplace=True),
            nn.LocalResponseNorm(size=5,alpha=0.0001,beta=0.75,k=1.0),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        # Third Layer: Conv + ReLU + MaxPool
        self.layer3 = nn.Sequential (
            nn.Conv2d(in_channels=20, out_channels=30, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        # Fourth Layer: FC + ReLU
        self.layer4 = nn.Sequential (
            nn.Linear(in_features=30*3*3, out_features=48),
            nn.ReLU(inplace=True)
        )
        # Fifth Layer: FC
        self.layer5 = nn.Sequential (
            nn.Linear(in_features=48, out_features=2)
        )

    def forward (self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = torch.flatten(x, start_dim=1)
        x = self.layer4(x)
        x = self.layer5(x)
        return x

In [3]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinando na unidade: {device}")

model = mAlex().to(device)

loss_fn = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005)

model.train()

running_loss = 0.0

for batch_idx, (images, labels) in enumerate(train_dataloader):

    images = images.to(device)
    labels = labels.to(device)
    
    optimizer.zero_grad()
    
    outputs = model(images)
    
    loss = loss_fn(outputs, labels)

    loss.backward()
    
    optimizer.step()
    
    running_loss += loss.item()

epoch_loss = running_loss / len(train_dataloader)

print(f"Epoch Loss Média: {epoch_loss:.4f}")

Treinando na unidade: cpu
Epoch Loss Média: 0.6744
